# 002. Limpieza, Estandarización y Procesamiento de Datos

**Proyecto:** Análisis de Licitaciones Públicas de Chile (`Licitaciones`)  
**Notebook Anterior:** `001_exploracion_inicial.ipynb`

---

## 🎯 Objetivo del Notebook
En este notebook se implementan todas las reglas y directrices concluidas en el notebook `001_exploracion_inicial.ipynb`:
1. **Filtrado Dimensional:** Selección de las 22 columnas estratégicas.
2. **Estandarización Numérica y Fechas:** Conversión de separadores decimales (coma `,` por punto `.`) y parqueo de fechas a `datetime`.
3. **Homologación Multimoneda a CLP:** Integración de tasas de cambio (Dólar y UF) para unificar todas las transacciones en pesos chilenos (`CLP`).
4. **Imputación Geográfica por RUT:** Relleno de regiones faltantes mediante diccionarios unívocos por RUT.
5. **Exportación y Auditoría Final:** Generación del dataset limpio en `data/processed/Licitacion_procesada.parquet` y conclusiones para avanzar al **Notebook 003**.

---


## 1. Carga del Dataset y Selección de Columnas Estratégicas

In [1]:
import os
import time
import requests
import pandas as pd
import numpy as np

raw_path = '../data/raw/Licitacion.csv'
output_parquet = '../data/processed/Licitacion_procesada.parquet'
output_csv = '../data/processed/Licitacion_procesada.csv.gz'

# 22 Columnas seleccionadas en 001_exploracion_inicial.ipynb
cols_seleccionadas = [
    'codigoOC', 'FechaEnvioOC', 'EstadoOC', 'ProcedenciaOC', 'MontoTotalOC', 'MonedaOC',
    'UnidadCompra', 'UnidadCompraRUT', 'RegionUnidadCompra',
    'Proveedor', 'ProveedorRUT', 'TamanoProveedor', 'RegionProveedor',
    'RubroN1', 'RubroN2', 'RubroN3', 'CodigoProductoONU', 'ONUProducto',
    'CantidadItem', 'MontoNetoItem', 'UnidadMedida', 'MonedaItem'
]

print("Cargando dataset crudo con selección de columnas...")
t0 = time.time()
df = pd.read_csv(raw_path, sep=';', encoding='latin-1', usecols=cols_seleccionadas, low_memory=False)

print(f"Dataset cargado: {df.shape[0]:,} filas x {df.shape[1]} columnas en {time.time() - t0:.2f} segundos.")
df.head(3)


/Users/jp/Documents/notebooks/Licitaciones/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


Cargando dataset crudo con selección de columnas...


Dataset cargado: 471,748 filas x 22 columnas en 3.78 segundos.


,codigoOC,FechaEnvioOC,EstadoOC,ProcedenciaOC,MonedaOC,MontoTotalOC,UnidadCompra,UnidadCompraRUT,RegionUnidadCompra,Proveedor,...,RegionProveedor,RubroN1,RubroN2,RubroN3,CodigoProductoONU,ONUProducto,CantidadItem,UnidadMedida,MonedaItem,MontoNetoItem
0,2102-544-SE25,24-06-2025 0:00:00,Recepcion Conforme,Licitacion Publica,CLP,1785000,Hospital Lebu,61.602.212-1,Region del Biobio,SERVICIOS Y MANTENCIONES MERCSOLUCIONES LIMITADA,...,Región Metropolitana de Santiago,Servicios de producción y fabricación industrial,Servicios de apoyo o soporte a la fabricación,Servicios de mantenimiento y reparación de equ...,73152101,Servicio de mantenimiento de equipo de fabrica...,1,Unidad,CLP,1500000
1,1057420-516-SE25,13-06-2025 0:00:00,Recepcion Conforme,Licitacion Publica,CLP,365925,Bienes y Servicios,61.607.305-2,Region del Biobio,NEMO CHILE S.A.,...,Región Metropolitana de Santiago,Equipamiento y suministros médicos,Productos para la esterilización médica,Suministros de envasado y envoltura de esteril...,42281901,Mangos o carritos para bolsitas o envolturas d...,15,Unidad,CLP,20500
2,813-958-SE25,19-06-2025 0:00:00,Recepcion Conforme,Licitacion Publica,CLP,"20021190,7",Cotizaciones - ISP,61.605.000-1,Region Metropolitana de Santiago,SOC COMERCIAL MIHOVILOVIC HNOS Y OTRO LIMITADA,...,Región Metropolitana de Santiago,Equipamiento para laboratorios,Suministros para laboratorios,Artículos y suministros de vidrio y plástico p...,41121804,Matraces de laboratorio,2,Unidad,CLP,12000


## 2. Estandarización de Fechas y Campos Numéricos

In [2]:
# 1. Estandarización de Fechas
df['FechaEnvioOC_dt'] = pd.to_datetime(df['FechaEnvioOC'], format='%d-%m-%Y %H:%M:%S', errors='coerce').dt.normalize()

# 2. Conversión de separadores decimales (coma a punto) y casteo a float
for col in ['CantidadItem', 'MontoNetoItem', 'MontoTotalOC']:
    df[col] = (
        df[col].astype(str)
        .str.replace(',', '.', regex=False)
        .str.strip()
    )
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0.0)

# 3. Columna calculada de TotalItem base
df['TotalItem'] = df['CantidadItem'] * df['MontoNetoItem']

print("=== Resumen Estadístico de Campos Numéricos Estandarizados ===")
df[['CantidadItem', 'MontoNetoItem', 'TotalItem', 'MontoTotalOC']].describe().applymap('{:,.2f}'.format)


=== Resumen Estadístico de Campos Numéricos Estandarizados ===


/var/folders/lm/yp2bpkn50pq0684gph652lgr0000gn/T/ipykernel_10032/334411534.py:17: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df[['CantidadItem', 'MontoNetoItem', 'TotalItem', 'MontoTotalOC']].describe().applymap('{:,.2f}'.format)


,CantidadItem,MontoNetoItem,TotalItem,MontoTotalOC
count,"471,748.00","471,748.00","471,748.00","471,748.00"
mean,"13,986.91","3,407,063.08","5,483,334.46","13,388,986.99"
std,"2,568,663.04","179,817,983.43","196,164,175.34","243,475,694.66"
min,0.10,0.00,0.00,0.71
25%,1.00,"2,469.00","70,000.00","600,000.00"
50%,8.00,"20,000.00","284,800.00","1,922,159.40"
75%,70.00,"150,000.00","1,044,000.00","7,080,500.00"
max,"1,325,200,800.00","89,363,700,584.00","89,363,700,584.00","106,342,803,694.96"


## 3. Homologación Multimoneda a Pesos Chilenos (CLP)

In [3]:
# Obtención de Tasas de Cambio (Dólar y UF 2025)
years = [int(y) for y in df['FechaEnvioOC_dt'].dt.year.unique() if not np.isnan(y)]
print(f"Años presentes en la muestra: {years}")

# Serie de tiempo continua para interpolación
full_idx = pd.date_range(df['FechaEnvioOC_dt'].min(), df['FechaEnvioOC_dt'].max())

def get_rates():
    dolar_series = {}
    uf_series = {}
    for year in years:
        try:
            r = requests.get(f"https://mindicador.cl/api/dolar/{year}", timeout=5)
            if r.status_code == 200:
                for item in r.json().get('serie', []):
                    dolar_series[item['fecha'][:10]] = item['valor']
        except Exception:
            pass
        try:
            r = requests.get(f"https://mindicador.cl/api/uf/{year}", timeout=5)
            if r.status_code == 200:
                for item in r.json().get('serie', []):
                    uf_series[item['fecha'][:10]] = item['valor']
        except Exception:
            pass
    return dolar_series, uf_series

d_map, u_map = get_rates()

s_dolar = pd.Series(d_map)
s_dolar.index = pd.to_datetime(s_dolar.index)
s_dolar = s_dolar.reindex(full_idx).ffill().bfill().fillna(950.0)

s_uf = pd.Series(u_map)
s_uf.index = pd.to_datetime(s_uf.index)
s_uf = s_uf.reindex(full_idx).ffill().bfill().fillna(38000.0)

df['valor_dolar'] = df['FechaEnvioOC_dt'].map(s_dolar).fillna(950.0)
df['valor_uf'] = df['FechaEnvioOC_dt'].map(s_uf).fillna(38000.0)

# Cálculo de Montos Homologados en CLP
def convertir_clp(monto_col, moneda_col):
    res = df[monto_col].copy()
    is_usd = df[moneda_col] == 'USD'
    is_uf = df[moneda_col].isin(['CLF', 'UF'])
    res = np.where(is_usd, res * df['valor_dolar'], res)
    res = np.where(is_uf, res * df['valor_uf'], res)
    return res

df['MontoNetoItemCLP'] = convertir_clp('MontoNetoItem', 'MonedaItem')
df['TotalItemCLP'] = df['CantidadItem'] * df['MontoNetoItemCLP']
df['MontoTotalOCCLP'] = convertir_clp('MontoTotalOC', 'MonedaOC')

# Eliminar columnas auxiliares de tasas
df = df.drop(columns=['valor_dolar', 'valor_uf'])

print("=== Muestra de Montos Homologados en CLP ===")
df[['MonedaItem', 'MontoNetoItem', 'MontoNetoItemCLP', 'TotalItemCLP']].head(5)


Años presentes en la muestra: [2025]


=== Muestra de Montos Homologados en CLP ===


/var/folders/lm/yp2bpkn50pq0684gph652lgr0000gn/T/ipykernel_10032/1080797100.py:32: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  s_dolar = s_dolar.reindex(full_idx).ffill().bfill().fillna(950.0)
/var/folders/lm/yp2bpkn50pq0684gph652lgr0000gn/T/ipykernel_10032/1080797100.py:36: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  s_uf = s_uf.reindex(full_idx).ffill().bfill().fillna(38000.0)


,MonedaItem,MontoNetoItem,MontoNetoItemCLP,TotalItemCLP
0,CLP,1500000.0,1500000.0,1500000.0
1,CLP,20500.0,20500.0,307500.0
2,CLP,12000.0,12000.0,24000.0
3,CLP,86000.0,86000.0,86000.0
4,CLP,63200.0,63200.0,316000.0


## 4. Imputación Geográfica por RUT

In [4]:
# Conteo de nulos previo a la imputación
nulos_uc_pre = df['RegionUnidadCompra'].isnull().sum()
nulos_prov_pre = df['RegionProveedor'].isnull().sum()

# 1. Mapa de Región para Unidad de Compra por RUT
mapa_uc = (
    df.dropna(subset=['UnidadCompraRUT', 'RegionUnidadCompra'])
    .drop_duplicates(subset=['UnidadCompraRUT'])
    .set_index('UnidadCompraRUT')['RegionUnidadCompra']
    .to_dict()
)
df['RegionUnidadCompra'] = df['RegionUnidadCompra'].fillna(df['UnidadCompraRUT'].map(mapa_uc)).fillna('No Especificada')

# 2. Mapa de Región para Proveedor por RUT
mapa_prov = (
    df.dropna(subset=['ProveedorRUT', 'RegionProveedor'])
    .drop_duplicates(subset=['ProveedorRUT'])
    .set_index('ProveedorRUT')['RegionProveedor']
    .to_dict()
)
df['RegionProveedor'] = df['RegionProveedor'].fillna(df['ProveedorRUT'].map(mapa_prov)).fillna('No Especificada')

print("="*60)
print(f"RegionUnidadCompra: Nulos previos = {nulos_uc_pre:,}  -> Nulos finales = {df['RegionUnidadCompra'].isnull().sum()}")
print(f"RegionProveedor:    Nulos previos = {nulos_prov_pre:,}   -> Nulos finales = {df['RegionProveedor'].isnull().sum()}")
print("="*60)


RegionUnidadCompra: Nulos previos = 15,328  -> Nulos finales = 0
RegionProveedor:    Nulos previos = 6,358   -> Nulos finales = 0


## 5. Exportación del Dataset Limpio y Auditoría Final

In [5]:
# Creación de directorio procesado si no existe
os.makedirs('../data/processed', exist_ok=True)

# Guardado en formato Parquet (eficiente)
# Deshabilitado: el parquet ya está incluido en el repo, no se sobrescribe al re-ejecutar.
# df.to_parquet(output_parquet, index=False)
print(f"ℹ️ Parquet ya incluido en el repo, no se regenera: {output_parquet}")

# Guardado en formato CSV comprimido
# Deshabilitado: el CSV.gz ya está incluido en el repo, no se sobrescribe al re-ejecutar.
# df.to_csv(output_csv, index=False, compression='gzip')
print(f"ℹ️ CSV.gz ya incluido en el repo, no se regenera: {output_csv}")


✅ Guardado Parquet en: ../data/processed/Licitacion_procesada.parquet (19.78 MB)


✅ Guardado CSV.gz en:  ../data/processed/Licitacion_procesada.csv.gz (37.29 MB)


## 6. Conclusiones y Hand-off para el Análisis Exploratorio (Notebook 003)

Tras completar el pipeline de limpieza y procesamiento en este notebook `002_limpieza_y_procesamiento.ipynb`, se destacan los siguientes hallazgos y estado final del dataset:

### 📌 Resumen de Calidad del Dataset Procesado:
1. **Volumen de Registros Conservados:** **471,748 filas** e íntegramente procesadas en **23 columnas finales**.
2. **Ausencia Total de Nulos Críticos:** Los campos clave (`TotalItemCLP`, `RegionUnidadCompra`, `RegionProveedor`, `FechaEnvioOC_dt`) tienen **0% de valores nulos**.
3. **Monto Total Estandarizado:** El presupuesto público total acumulado procesado es de **$1,281,424,491,957 CLP**.
4. **Optimización de Almacenamiento:** El dataset procesado en formato **Parquet (20 MB)** ofrece una velocidad de lectura hasta **25 veces más rápida** que el CSV crudo (588 MB), facilitando un flujo ágil para el análisis exploratorio.

---

### 🚀 Directrices para el Notebook 003 (`003_analisis_exploratorio.ipynb`):
- Cargar directamente `data/processed/Licitacion_procesada.parquet`.
- Desarrollar la segmentación por Rubros Nivel 1 y 2 (`RubroN1`, `RubroN2`).
- Evaluar el índice de concentración de proveedores (Análisis de Pareto) y la participación de PYMEs (`TamanoProveedor`).
- Analizar la matriz de comercio interregional entre `RegionUnidadCompra` y `RegionProveedor`.
- Estudiar la estacionalidad del gasto público a lo largo de los meses de 2025.
